## <a href="https://cursos.alura.com.br/course/langchain-python-ferramentas-llm-openai/task/156167"><b>Gerenciando a memória de uma conversa com as bibliotecas do LangChain</b></a><br/>

Estou nessa parte do curso (https://cursos.alura.com.br/course/langchain-python-ferramentas-llm-openai/task/156168?b2cUser=true)

In [8]:
%pip install -qr requirements.txt

Note: you may need to restart the kernel to use updated packages.


## <b>ConversationBufferMemory</b>

#### <b>PASSO 1 - IMPORTS e CRIAÇÃO DA LLM</b>

In [11]:
from langchain_openai import ChatOpenAI
from os import getenv
from dotenv import load_dotenv # CARREGA A VARIÁVEL DE AMBIENTE OPENAI_KEY LIDA DO ARQUIVO .env

load_dotenv() # CARREGANDO O ARQUIVO COM A OPENAI_KEY

llm = ChatOpenAI( # INSTANCIANDO A LLM
                    model="gpt-5-mini",                    
                    # 1 - OBTENDO A API KEY POR MEIO DA VARIÁVEL DE AMBIENTE OPENAI_KEY. QUE VAI FICAR ARMAZENADA NO ARQUIVO .env.
                    # 2 - AINDA É NECESSÁRIO CARREGAR ESSE ARQUIVO. VER NA PRIMEIRA CÉLULA DO NOTEBOOK
                    api_key=getenv("OPENAI_KEY")                    
                )

#### <b> PASSO 2 - CRIANDO UM EXEMPLO DE CONVERSAÇÃO DE CHAT<b>

In [ ]:
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

mensagens = [
        "Quero visitar um lugar no Brasil famoso por suas praias e cultura. Pode me recomendar?",
        "Qual é o melhor período do ano para visitar em termos de clima?",
        "Quais tipos de atividades ao ar livre estão disponíveis?",
        "Alguma sugestão de acomodação eco-friendly por lá?",
        "Cite outras 20 cidades com características semelhantes às que descrevemos até agora. Rankeie por mais interessante, incluindo no meio a que você já sugeriu.",
        "Na primeira cidade que você sugeriu lá atrás, quero saber 5 restaurantes para visitar. Responda somente o nome da cidade e o nome dos restaurantes.",
]

longa_conversa = ""
for mensagem in mensagens:
        longa_conversa = f"Usuário: {mensagem}\n"
        longa_conversa += f"IA: "
        
        prompt_template = PromptTemplate(
                                                template=longa_conversa,
                                                input_variables=[""]
                                        ) 


        cadeia = prompt_template | llm | StrOutputParser()
        resposta = cadeia.invoke({})

        longa_conversa += resposta + "\n"
        
        print(longa_conversa) # A IA NÃO VAI CONSEGUIR RESPONDER A ÚLTIMA PERGUNTA, POR CAUSA DA FALTA DE MEMÓRIA

Usuário: Quero visitar um lugar no Brasil famoso por suas praias e cultura. Pode me recomendar?
IA: Claro — posso recomendar alguns lugares ótimos no Brasil que combinam praias lindas e muita cultura. Aqui vão três sugestões, com um resumo rápido para você escolher conforme seu estilo:

1) Salvador (BA)
- Por que ir: é um dos melhores destinos para cultura afro-brasileira — Pelourinho, candomblé, capoeira, música (axé, samba-reggae) e gastronomia baiana.  
- Praias: Porto da Barra, Itapuã, Flamengo; passeios fáceis para Praia do Forte e Morro de São Paulo.  
- Melhor época: setembro a março (Carnaval em fevereiro/março é espetacular, mas muito cheio).  
- Dica: hospede-se na Barra/Ondina para praia e fácil acesso ao centro histórico; tenha cuidado com pertences em áreas turísticas à noite.

2) Rio de Janeiro (RJ)
- Por que ir: ícones nacionais (Cristo Redentor, Pão de Açúcar), vida cultural vibrante (Lapa, samba, museus) e festas.  
- Praias: Copacabana, Ipanema, Leblon; também trilhas

#### <b> PASSO 3 - CRIANDO CONVERSAÇÃO COM MEMORIA<b>

In [ ]:
from langchain_core.globals import set_debug

set_debug(True)

mensagens = [
        "Quero visitar um lugar no Brasil famoso por suas praias e cultura. Pode me recomendar?",
        "Qual é o melhor período do ano para visitar em termos de clima?",
        "Quais tipos de atividades ao ar livre estão disponíveis?",
        "Alguma sugestão de acomodação eco-friendly por lá?",
        "Cite outras 20 cidades com características semelhantes às que descrevemos até agora. Rankeie por mais interessante, incluindo no meio a que você já sugeriu.",
        "Na primeira cidade que você sugeriu lá atrás, quero saber 5 restaurantes para visitar. Responda somente o nome da cidade e o nome dos restaurantes.",
]



#### O tipo de memória que vamos usar

In [15]:
from langchain.memory import ConversationBufferMemory

# O TIPO DE MEMÓRIA QUE VAMOS USAR
memory = ConversationBufferMemory()

#### A conversação

In [18]:
from langchain.chains import ConversationChain

conversation = ConversationChain(
    llm=llm,
    verbose=True,
    memory=memory  # INFORMO QUE A CONVERSA VAI TER MEMÓRIA
)

In [ ]:
for mensagem in mensagens:       
        # NÃO PRECISO MAIS MONTAR A CONVERSA, POIS A MEMÓRIA VAI CUIDAR DISSO
        resposta = conversation.predict(input=mensagem) # NESSA CONVERSAÇÃO, PREDIZ COMO ELA VAI CONTINUAR PARA MIM
        
        print(resposta)
        
        

#### <b>É interessante observar a variável "history"</b>

<font color="green"><b>[chain/start]</b></font><b> [chain:ConversationChain] Entering Chain run with input:</b><br/>
{
  <ul><b>"input": "Quero visitar um lugar no Brasil famoso por suas praias e cultura. Pode me recomendar?",</b></ul>
  <ul>"history": ""</ul>
}<br/>
<font color="green"><b>[llm/start]</b></font><b> [chain:ConversationChain > llm:ChatOpenAI] Entering LLM run with input:</b><br/>
{<br/>
  <ul>"prompts": [</ul>
    <ul>"Human: The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does<br/>not know the answer to a question, it truthfully says it does not know.\n\n<b>Current conversation:\n\nHuman: Quero visitar um lugar no Brasil famoso por suas praias e <br/>cultura. Pode me recomendar?\nAI:</b>"</ul>
  ]
}

<font color="cyan"><b>[chain/end]</b></font><b> [chain:ConversationChain] [29.79s] Exiting Chain run with output:</b><br/>
{<br/>
  <ul>"response": "Ótimo — o Brasil tem muitas opções que combinam praias e cultura. Vou sugerir cinco destinos com perfis diferentes (praia + cultura), com destaques práticos para você escolher conforme seu estilo:\n\n1) Salvador (Bahia)\n- Por que ir: berço da cultura afro-brasileira, música (samba-reggae, axé), candomblé, culinária típica.\n- Praias: Porto da Barra, Flamengo, Itapuã; praias melhores para banho nas proximidades (Praia do Forte, Morro de São Paulo).\n- Pontos culturais: Pelourinho (centro histórico), Elevador Lacerda, igrejas barrocas, festas de rua.\n- Comida: acarajé, moqueca, vatapá, cocada, caranguejo.\n- Melhor época: setembro a março (clima quente); Carnaval e festas de largo são espetaculares, mas cheios.\n- Duração recomendada: 3–5 dias (mais se for explorar ilhas e praias próximas).\n- Acesso: voos diretos para Salvador (SSA). Transporte local via táxi/ride-hailing e passeios organizados.\n- Dica de segurança: cuidado com pertences em áreas turísticas muito cheias; prefira táxis confiáveis à noite.\n\n2) Rio de Janeiro (RJ)\n- Por que ir: combinação clássica de praias urbanas famosas e forte cena cultural (música, museus, vida noturna).\n- Praias: Copacabana, Ipanema, Arpoador; praias mais tranquilas na Zona Oeste (Recreio) e ilhas próximas.\n- Pontos culturais: Cristo Redentor, Pão de Açúcar, Museu do Amanhã, Lapa (samba e bares).\n- Comida: feijoada, churrasco, frutos do mar, caldo de cana com pastel.\n- Melhor época: abril–novembro (menor chance de chuva); Carnaval é imperdível, mas lotado e caro.\n- Duração: 3–6 dias.\n- Acesso: voos para Galeão (GIG) ou Santos Dumont (SDU); transporte público e aplicativo de transporte amplamente disponíveis.\n- Dica de segurança: evite exibir objetos de valor em praias e transporte; informe-se sobre bairros para hospedagem (Ipanema, Leblon, Lapa, Botafogo são opções populares).\n\n3) Florianópolis (Santa Catarina)\n- Por que ir: ilha com praias incríveis, forte cultura açoriana, boa cena de surf e vida noturna.\n- Praias: Joaquina (surf), Mole (jovem), Campeche, Lagoinha do Leste (trilha).\n- Pontos culturais: centros históricos e festas tradicionais açorianas; ótimos frutos do mar.\n- Comida: sequência de camarão, ostras (na região de Florianópolis/Palhoça).\n- Melhor época: dezembro–março (alta temporada); setembro–novembro e março–abril para menos lotação.\n- Duração: 3–7 dias.\n- Acesso: voos para Florianópolis (FLN) + aluguel de carro é útil para explorar a ilha.\n- Dica: alugue carro para aproveitar praias mais isoladas; reserve hospedagem com antecedência na alta temporada.\n\n4) Recife + Olinda (Pernambuco)\n- Por que ir: mix de praias urbanas e patrimônio histórico colonial com muita música local (frevo, maracatu).\n- Praias: Boa Viagem (Recife), praias do litoral sul e norte do estado.\n- Pontos culturais: centro histórico de Olinda (patrimônio), festivais de rua, museus.\n- Comida: tapioca, bolo de rolo, frutos do mar, carne de sol.\n- Melhor época: setembro–março; Carnaval em Olinda é um dos mais autênticos do Brasil.\n- Duração: 2–5 dias (Olinda pode ser visitada em um bate-volta desde Recife).\n- Acesso: voos para Recife (REC). Transporte local via táxi/ride-hailing; passeios guiados em Olinda.\n- Dica cultural: caminhar pelas ladeiras de Olinda ao pôr do sol é memorável — use calçados confortáveis.\n\n5) Jericoacoara (Ceará)\n- Por que ir: vila praiana famosa por dunas espetaculares, pôr do sol nas dunas e forte cultura de windsurf/kitesurf.\n- Praias: Praia de Jericoacoara, Praia do Preá (próxima para kitesurf).\n- Pontos culturais: vila rústica com bares, forró e festas ao redor do vilarejo; muito contato com a natureza.\n- Comida: peixes e frutos do mar frescos, tapioca.\n- Melhor época: julho–dezembro (vento constante para esportes); dezembro–março para mar mais calmo.\n- Duração: 3–5 dias.\n- Acesso: avião até Fortaleza (FOR) ou Jericoacoara (JJD dependendo da época), + transfer/4x4 até a vila.\n- Dica logística: leve dinheiro em espécie para alguns estabelecimentos; estradas de areia exigem transfers.\n\nQuer que eu foque em um desses (ex.: Salvador ou Rio) e monte um roteiro detalhado de 3–5 dias com hotéis, restaurantes e transporte, ou prefere opções mais fora do circuito (ilhas menores, pousadas) ou orçamento específico?"</ul>
}

<font color="green"><b>[chain/start]</b></font> [chain:ConversationChain] Entering Chain run with input:
{
  <ul><b>"input": "Qual é o melhor período do ano para visitar em termos de clima?",</b></ul>
  <ul>"history": <b>"Human: Quero visitar um lugar no Brasil famoso por suas praias e cultura. Pode me recomendar?</b><i>\nAI: Ótimo — o Brasil tem muitas opções que combinam praias e cultura. Vou sugerir cinco destinos com perfis diferentes (praia + cultura), com destaques práticos para você escolher conforme seu estilo:\n\n1) Salvador (Bahia)\n- Por que ir: berço da cultura afro-brasileira, música (samba-reggae, axé), candomblé, culinária típica.\n- Praias: Porto da Barra, Flamengo, Itapuã; praias melhores para banho nas proximidades (Praia do Forte, Morro de São Paulo).\n- Pontos culturais: Pelourinho (centro histórico), Elevador Lacerda, igrejas barrocas, festas de rua.\n- Comida: acarajé, moqueca, vatapá, cocada, caranguejo.\n- Melhor época: setembro a março (clima quente); Carnaval e festas de largo são espetaculares, mas cheios.\n- Duração recomendada: 3–5 dias (mais se for explorar ilhas e praias próximas).\n- Acesso: voos diretos para Salvador (SSA). Transporte local via táxi/ride-hailing e passeios organizados.\n- Dica de segurança: cuidado com pertences em áreas turísticas muito cheias; prefira táxis confiáveis à noite.\n\n2) Rio de Janeiro (RJ)\n- Por que ir: combinação clássica de praias urbanas famosas e forte cena cultural (música, museus, vida noturna).\n- Praias: Copacabana, Ipanema, Arpoador; praias mais tranquilas na Zona Oeste (Recreio) e ilhas próximas.\n- Pontos culturais: Cristo Redentor, Pão de Açúcar, Museu do Amanhã, Lapa (samba e bares).\n- Comida: feijoada, churrasco, frutos do mar, caldo de cana com pastel.\n- Melhor época: abril–novembro (menor chance de chuva); Carnaval é imperdível, mas lotado e caro.\n- Duração: 3–6 dias.\n- Acesso: voos para Galeão (GIG) ou Santos Dumont (SDU); transporte público e aplicativo de transporte amplamente disponíveis.\n- Dica de segurança: evite exibir objetos de valor em praias e transporte; informe-se sobre bairros para hospedagem (Ipanema, Leblon, Lapa, Botafogo são opções populares).\n\n3) Florianópolis (Santa Catarina)\n- Por que ir: ilha com praias incríveis, forte cultura açoriana, boa cena de surf e vida noturna.\n- Praias: Joaquina (surf), Mole (jovem), Campeche, Lagoinha do Leste (trilha).\n- Pontos culturais: centros históricos e festas tradicionais açorianas; ótimos frutos do mar.\n- Comida: sequência de camarão, ostras (na região de Florianópolis/Palhoça).\n- Melhor época: dezembro–março (alta temporada); setembro–novembro e março–abril para menos lotação.\n- Duração: 3–7 dias.\n- Acesso: voos para Florianópolis (FLN) + aluguel de carro é útil para explorar a ilha.\n- Dica: alugue carro para aproveitar praias mais isoladas; reserve hospedagem com antecedência na alta temporada.\n\n4) Recife + Olinda (Pernambuco)\n- Por que ir: mix de praias urbanas e patrimônio histórico colonial com muita música local (frevo, maracatu).\n- Praias: Boa Viagem (Recife), praias do litoral sul e norte do estado.\n- Pontos culturais: centro histórico de Olinda (patrimônio), festivais de rua, museus.\n- Comida: tapioca, bolo de rolo, frutos do mar, carne de sol.\n- Melhor época: setembro–março; Carnaval em Olinda é um dos mais autênticos do Brasil.\n- Duração: 2–5 dias (Olinda pode ser visitada em um bate-volta desde Recife).\n- Acesso: voos para Recife (REC). Transporte local via táxi/ride-hailing; passeios guiados em Olinda.\n- Dica cultural: caminhar pelas ladeiras de Olinda ao pôr do sol é memorável — use calçados confortáveis.\n\n5) Jericoacoara (Ceará)\n- Por que ir: vila praiana famosa por dunas espetaculares, pôr do sol nas dunas e forte cultura de windsurf/kitesurf.\n- Praias: Praia de Jericoacoara, Praia do Preá (próxima para kitesurf).\n- Pontos culturais: vila rústica com bares, forró e festas ao redor do vilarejo; muito contato com a natureza.\n- Comida: peixes e frutos do mar frescos, tapioca.\n- Melhor época: julho–dezembro (vento constante para esportes); dezembro–março para mar mais calmo.\n- Duração: 3–5 dias.\n- Acesso: avião até Fortaleza (FOR) ou Jericoacoara (JJD dependendo da época), + transfer/4x4 até a vila.\n- Dica logística: leve dinheiro em espécie para alguns estabelecimentos; estradas de areia exigem transfers.\n\nQuer que eu foque em um desses (ex.: Salvador ou Rio) e monte um roteiro detalhado de 3–5 dias com hotéis, restaurantes e transporte, ou prefere opções mais fora do circuito (ilhas menores, pousadas) ou orçamento específico?"</i></ul>
}

#### Concatenou a pergunta e resposta anterior, a próxima pergunta: ""input": "Qual é o melhor período do ano para visitar em termos de clima?""

<font color="cyan"><b>[chain/end]</b></font>[chain:ConversationChain] [29.34s] Exiting Chain run with output:<br/>
{<br/>
  <ul>"response": "Boa pergunta — “melhor época” depende do que você espera (praia calma, surf/kitesurf, festas, menos chuva). Abaixo eu resumo, destino a destino, os períodos mais favoráveis em termos de clima e o porquê:\n\n- Salvador (BA)\n  - Melhor época: setembro–março — mais seco e quente, ótimo para praia e eventos ao ar livre.\n  - Evitar: abril–julho (mais chuvas, especialmente em maio/junho).\n  - Observação: Carnaval (fev/mar) é espetacular, mas muito cheio e caro.\n\n- Rio de Janeiro (RJ)\n  - Melhor época geral: abril–novembro — menos chuva, temperaturas agradáveis.\n  - Melhor para praia quente: dezembro–fevereiro — verão, muito calor (também há mais chuva rápida e multidões).\n  - Evitar se quer clima seco e tranquilo: alta temporada de verão e feriados prolongados (Natal/Ano-Novo, Carnaval).\n\n- Florianópolis (SC)\n  - Melhor para praia: dezembro–fevereiro — verão com temperaturas ideais para banho.\n  - Bons meses com menos lotação: setembro–novembro e março–abril (ótimos “shoulders”).\n  - Evitar: junho–agosto se quer praia (mais frio e mar agitado).\n\n- Recife + Olinda (PE)\n  - Melhor época: setembro–março — mais seco e ensolarado.\n  - Período de chuvas: abril–julho (pico em maio/junho).\n  - Carnaval em Olinda (fev) é culturalmente riquíssimo, mas muito movimentado.\n\n- Jericoacoara (CE)\n  - Melhor para wind/kitesurf: julho–dezembro — ventos constantes e céu geralmente limpo.\n  - Melhor para mar mais calmo e banho: dezembro–março.\n  - Chuva: período de chuva é pequeno (fev–abr costuma ter mais precipitação).\n\nDicas gerais\n- Temporadas de festas (Carnaval, Natal/Ano-Novo) têm clima bom em muitos destinos, mas preços e multidões sobem muito — reserve com antecedência se planeja ir nessas datas.\n- Os “shoulders” (set–nov e mar–mai) costumam ser a melhor combinação entre bom clima, menos turistas e preços melhores.\n- O clima pode variar ano a ano; verifique previsão estendida antes de fechar a viagem.\n\nQuer que eu recomende o melhor mês conforme seu objetivo (ex.: praias calmas, esportes aquáticos ou festa cultural)? Posso também montar um roteiro com datas específicas."</ul>
}

<b>E agora não esqueceu qual foi a primeira cidade</b>

<font color="green"><b>[chain/start]</b></font> [chain:ConversationChain] Entering Chain run with input:<br/>
{
  <ul>"input": "Na primeira cidade que você sugeriu lá atrás, quero saber 5 restaurantes para visitar. Responda somente o nome da cidade e o nome dos restaurantes.",</ul>
.
.
.
.
.
}

<b>Salvador — Amado, Casa de Tereza, Yemanjá, Criolina, Restaurante do Senac</b>

Estou nessa parte do curso (https://cursos.alura.com.br/course/langchain-python-ferramentas-llm-openai/task/156168?b2cUser=true)

## <a href="https://cursos.alura.com.br/course/langchain-python-ferramentas-llm-openai/task/156168?b2cUser=true"><b>ConversationBufferWindow</b></a>